In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/processed/df_tripdata_engineered.csv')
df.head()

,ID_CODE,SURVEY_TIMESTAMP,FFDAYS2,HRSF,KOD,MODE_F,CNTRBTRS,IMP_REC,ATLANTIC COD,ATLANTIC CROAKER,...,avg_water_level,water_level_rose,water_level_fell,was_sunrise,was_sunset,was_moonrise,was_moonset,moon_phase_age,sun_distance,moon_distance
0,1000920140505017,2014-05-05 17:00:00,0.0,6.0,wd,7.0,1.0,0.0,0.0,0.0,...,0.491667,1,1,0,0,1,0,5.633444,1.508852e+08,404508.533199
1,1000920140505018,2014-05-05 17:00:00,0.0,6.0,wd,7.0,1.0,0.0,0.0,0.0,...,0.491667,1,1,0,0,1,0,5.633444,1.508852e+08,404508.533199
2,1000920150620006,2015-06-20 16:00:00,0.0,4.5,we,8.0,1.0,0.0,0.0,0.0,...,0.442800,1,1,0,0,0,0,3.611222,1.520199e+08,397917.383118
3,1000920150627020,2015-06-27 17:00:00,3.0,7.0,we,8.0,0.0,0.0,0.0,0.0,...,1.628286,1,1,0,0,1,0,9.600111,1.520708e+08,400109.337422
4,1000920150711002,2015-07-11 11:00:00,1.0,2.0,we,8.0,1.0,0.0,0.0,0.0,...,2.077000,1,0,0,0,0,0,23.600111,1.520861e+08,372690.223940


In [13]:
df.columns

Index(['ID_CODE', 'SURVEY_TIMESTAMP', 'FFDAYS2', 'HRSF', 'KOD', 'MODE_F',
       'CNTRBTRS', 'IMP_REC', 'ATLANTIC COD', 'ATLANTIC CROAKER',
       'ATLANTIC HERRING', 'ATLANTIC MACKEREL', 'ATLANTIC MENHADEN',
       'BLACK SEA BASS', 'BLUEFISH', 'CHUB MACKEREL', 'CUNNER',
       'DOGFISH SHARK', 'GRAY SNAPPER', 'GRAY TRIGGERFISH', 'HICKORY SHAD',
       'LITTLE TUNNY', 'NORTHERN KINGFISH', 'NORTHERN PUFFER',
       'OYSTER TOADFISH', 'PINFISH', 'RED DRUM', 'RED HAKE', 'SCUP',
       'SEAROBIN GENUS', 'SKATE GENUS', 'SOUTHERN KINGFISH',
       'SPANISH MACKEREL', 'SPOT', 'SPOTTED SEATROUT', 'STRIPED BASS',
       'SUMMER FLOUNDER', 'TAUTOG', 'UNIDENTIFIED (SHARKS)',
       'UNIDENTIFIED SKATE OR RAY', 'WEAKFISH', 'WINDOWPANE',
       'WINTER FLOUNDER', 'avg_wind_speed', 'avg_wind_gust', 'wind_speed_rose',
       'wind_speed_fell', 'avg_air_pressure', 'air_pressure_rose',
       'air_pressure_fell', 'avg_air_temp_c', 'air_temp_rose', 'air_temp_fell',
       'avg_water_temp_c', 'water_tem

In [14]:
fishcols = ['ATLANTIC COD', 'ATLANTIC CROAKER',
       'ATLANTIC HERRING', 'ATLANTIC MACKEREL', 'ATLANTIC MENHADEN',
       'BLACK SEA BASS', 'BLUEFISH', 'CHUB MACKEREL', 'CUNNER',
       'DOGFISH SHARK', 'GRAY SNAPPER', 'GRAY TRIGGERFISH', 'HICKORY SHAD',
       'LITTLE TUNNY', 'NORTHERN KINGFISH', 'NORTHERN PUFFER',
       'OYSTER TOADFISH', 'PINFISH', 'RED DRUM', 'RED HAKE', 'SCUP',
       'SEAROBIN GENUS', 'SKATE GENUS', 'SOUTHERN KINGFISH',
       'SPANISH MACKEREL', 'SPOT', 'SPOTTED SEATROUT', 'STRIPED BASS',
       'SUMMER FLOUNDER', 'TAUTOG', 'UNIDENTIFIED (SHARKS)',
       'UNIDENTIFIED SKATE OR RAY', 'WEAKFISH', 'WINDOWPANE',
       'WINTER FLOUNDER']

In [15]:
df["total_catch"] = df[fishcols].sum(axis=1)

In [16]:
# Feature engineering and cleaning
df["has_catch"] = (df["total_catch"] > 0).astype(int)

# Drop species columns (leakage)
df.drop(columns=fishcols, inplace=True, errors="ignore")

# Temporal features
ts = pd.to_datetime(df["SURVEY_TIMESTAMP"])
df["year"] = ts.dt.year
df["month"] = ts.dt.month
df["day"] = ts.dt.day
df["hour"] = ts.dt.hour
df["dayofweek"] = ts.dt.dayofweek
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

# Cyclical encoding for periodic features (hour, month, dayofweek)
for col, period in [("hour", 24), ("month", 12), ("dayofweek", 7)]:
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)

# Interaction and biological features
# Effort
df["total_effort_hours"] = df["HRSF"] * df["CNTRBTRS"]

# Solunar activity
df["moon_illumination"] = np.sin(np.pi * df["moon_phase_age"] / 29.53) ** 2
df["major_minor_feeding"] = df[["was_sunrise", "was_sunset", "was_moonrise", "was_moonset"]].sum(axis=1)

# Weather fronts
df["storm_approaching"] = ((df["air_pressure_fell"] == 1) & (df["wind_speed_rose"] == 1)).astype(int)
df["cold_front"] = ((df["air_pressure_rose"] == 1) & (df["air_temp_fell"] == 1)).astype(int)

# Water and air conditions
df["temp_diff"] = df["avg_air_temp_c"] - df["avg_water_temp_c"]
df["temp_shock_magnitude"] = abs(df["temp_diff"])
df["water_is_moving"] = ((df["water_level_rose"] == 1) | (df["water_level_fell"] == 1)).astype(int)

# Wind and pressure
df["wind_x_pressure"] = df["avg_wind_speed"] * df["avg_air_pressure"]
df["gust_ratio"] = df["avg_wind_gust"] / (df["avg_wind_speed"] + 1e-6)

# One-hot encode KOD
df = pd.get_dummies(df, columns=["KOD"], drop_first=True)

# Remove leakage columns (except targets)
leakage_exact = {
    "catch_count",
    "species_caught",
    "num_species",
    "total_kept",
    "total_released",
}
leakage_keywords = ("catch", "species", "kept", "released")
leakage_cols = [
    c
    for c in df.columns
    if c not in {"has_catch", "total_catch"}
    and (c in leakage_exact or any(k in c.lower() for k in leakage_keywords))
]
leakage_cols = sorted(set(leakage_cols))
if leakage_cols:
    print(f"Dropping leakage columns: {leakage_cols}")
    df = df.drop(columns=leakage_cols, errors="ignore")

# Drop non-feature columns
df = df.drop(columns=["ID_CODE", "SURVEY_TIMESTAMP"], errors="ignore")

In [17]:
df.head(100)

,FFDAYS2,HRSF,MODE_F,CNTRBTRS,IMP_REC,avg_wind_speed,avg_wind_gust,wind_speed_rose,wind_speed_fell,avg_air_pressure,...,moon_illumination,major_minor_feeding,storm_approaching,cold_front,temp_diff,temp_shock_magnitude,water_is_moving,wind_x_pressure,gust_ratio,KOD_we
0,0.0,6.0,7.0,1.0,0.0,6.225000,7.441667,1,1,1010.825000,...,0.318190,1,1,1,1.475000,1.475000,1,6292.385625,1.195448,False
1,0.0,6.0,7.0,1.0,0.0,6.225000,7.441667,1,1,1010.825000,...,0.318190,1,1,1,1.475000,1.475000,1,6292.385625,1.195448,False
2,0.0,4.5,8.0,1.0,0.0,5.946667,7.013333,0,1,1018.053333,...,0.140478,0,0,1,0.353333,0.353333,1,6054.023822,1.179372,True
3,3.0,7.0,8.0,0.0,0.0,7.223810,8.509524,1,1,1019.280952,...,0.727266,1,1,1,-0.742857,0.742857,1,7363.091451,1.177983,True
4,1.0,2.0,8.0,1.0,0.0,3.566667,4.250000,0,1,1016.716667,...,0.347911,0,0,1,0.133333,0.133333,1,3626.289444,1.191588,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.0,3.5,7.0,3.0,0.0,10.975000,13.237500,1,1,1002.037500,...,0.999206,0,0,1,-4.887500,4.887500,1,10997.361563,1.206150,True
96,0.0,3.0,7.0,2.0,0.0,4.533333,5.255556,0,1,1010.511111,...,0.122651,0,0,0,-0.255556,0.255556,1,4580.983704,1.159313,False
97,0.0,3.0,5.0,1.0,0.0,9.588889,11.400000,1,1,1010.644444,...,0.194784,0,1,0,-0.477778,0.477778,1,9690.957284,1.188876,False
98,0.0,3.0,5.0,1.0,0.0,9.588889,11.400000,1,1,1010.644444,...,0.194784,0,1,0,-0.477778,0.477778,1,9690.957284,1.188876,False


In [18]:
df.columns

Index(['FFDAYS2', 'HRSF', 'MODE_F', 'CNTRBTRS', 'IMP_REC', 'avg_wind_speed',
       'avg_wind_gust', 'wind_speed_rose', 'wind_speed_fell',
       'avg_air_pressure', 'air_pressure_rose', 'air_pressure_fell',
       'avg_air_temp_c', 'air_temp_rose', 'air_temp_fell', 'avg_water_temp_c',
       'water_temp_rose', 'water_temp_fell', 'avg_water_level',
       'water_level_rose', 'water_level_fell', 'was_sunrise', 'was_sunset',
       'was_moonrise', 'was_moonset', 'moon_phase_age', 'sun_distance',
       'moon_distance', 'total_catch', 'has_catch', 'year', 'month', 'day',
       'hour', 'dayofweek', 'is_weekend', 'hour_sin', 'hour_cos', 'month_sin',
       'month_cos', 'dayofweek_sin', 'dayofweek_cos', 'total_effort_hours',
       'moon_illumination', 'major_minor_feeding', 'storm_approaching',
       'cold_front', 'temp_diff', 'temp_shock_magnitude', 'water_is_moving',
       'wind_x_pressure', 'gust_ratio', 'KOD_we'],
      dtype='object')

In [19]:
df.to_csv('../../data/processed/df_tripdata_engineered_with_total.csv', index=False)